# Notebook 03 – Metrics Đánh Giá Orchestrator & Agents

**Mục tiêu:** Tính toán đầy đủ các metrics để đánh giá hệ thống RL orchestrator.

## Metrics theo tầng

| Tầng | Metrics chính | Target |
|------|--------------|--------|
| **Orchestrator** | routing_accuracy, convergence_episode, q_value_stability | >85%, <200 ep |
| **Fundamental** | routing accuracy by agent, confidence | >80% |
| **Technical** | routing accuracy by agent | >85% |
| **Screening** | routing accuracy by agent | >75% |
| **Difficulty** | easy/medium/hard accuracy | >95%/>80%/>70% |

In [ ]:
import numpy as np
import pandas as pd
import random
from dataclasses import dataclass
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
random.seed(42)

DATA_DIR = '../data/'
df_orch = pd.read_csv(DATA_DIR + 'dataset_orchestrator.csv')
print(f'Loaded orchestrator dataset: {df_orch.shape}')

## 1. Setup: Agents & Orchestrator (tái sử dụng từ nb_01)

In [ ]:
@dataclass
class AgentResult:
    agent_name: str
    confidence: float
    result: dict
    execution_time_ms: float


class BaseAgent:
    name = ''
    keywords = []

    def _calculate_relevance(self, query: str) -> float:
        q = query.lower()
        matches = sum(1 for kw in self.keywords if kw.lower() in q)
        return min(1.0, matches * 0.25 + 0.1)

    def execute(self, query: str, context=None) -> AgentResult:
        raise NotImplementedError


class FundamentalAgent(BaseAgent):
    name = 'fundamental'
    keywords = [
        'P/E', 'PE', 'ROE', 'EPS', 'doanh thu', 'lợi nhuận', 'revenue',
        'báo cáo tài chính', 'financial report', 'cổ tức', 'dividend',
        'giá trị nội tại', 'intrinsic value', 'book value', 'P/B',
        'biên lợi nhuận', 'margin', 'nợ', 'debt', 'equity',
        'balance sheet', 'income statement', 'cash flow',
        'định giá', 'valuation', 'DCF', 'fundamental'
    ]

    def execute(self, query, context=None):
        rel = self._calculate_relevance(query)
        return AgentResult(self.name, rel,
                           {'P/E': round(random.uniform(8, 35), 2),
                            'recommendation': random.choice(['BUY', 'HOLD', 'SELL'])},
                           random.uniform(100, 500))


class TechnicalAgent(BaseAgent):
    name = 'technical'
    keywords = [
        'MA', 'SMA', 'EMA', 'RSI', 'MACD', 'Bollinger', 'xu hướng',
        'trend', 'support', 'resistance', 'hỗ trợ', 'kháng cự',
        'nến', 'candlestick', 'chart', 'biểu đồ', 'volume',
        'khối lượng', 'breakout', 'momentum', 'stochastic',
        'đường trung bình', 'moving average', 'tín hiệu',
        'signal', 'mua bán', 'entry', 'exit', 'technical'
    ]

    def execute(self, query, context=None):
        rel = self._calculate_relevance(query)
        return AgentResult(self.name, rel,
                           {'RSI': round(random.uniform(20, 80), 2),
                            'trend': random.choice(['UPTREND', 'DOWNTREND', 'SIDEWAYS'])},
                           random.uniform(50, 300))


class ScreeningAgent(BaseAgent):
    name = 'screening'
    keywords = [
        'lọc', 'sàng lọc', 'screen', 'filter', 'so sánh', 'compare',
        'top', 'best', 'tốt nhất', 'xếp hạng', 'ranking', 'sector',
        'ngành', 'industry', 'danh sách', 'list', 'tiêu chí',
        'criteria', 'tìm', 'search', 'find', 'which', 'nào',
        'recommend', 'gợi ý', 'đề xuất', 'suggestion'
    ]

    def execute(self, query, context=None):
        rel = self._calculate_relevance(query)
        return AgentResult(self.name, rel,
                           {'filtered_stocks': ['VNM', 'FPT', 'VCB']},
                           random.uniform(200, 800))


AGENT_REGISTRY = {
    'fundamental': FundamentalAgent(),
    'technical': TechnicalAgent(),
    'screening': ScreeningAgent(),
}
ACTION_SPACE = list(AGENT_REGISTRY.keys())
N_ACTIONS = len(ACTION_SPACE)


def extract_state(query: str) -> int:
    scores = [AGENT_REGISTRY[a]._calculate_relevance(query) for a in ACTION_SPACE]
    disc = [0 if s < 0.3 else (1 if s < 0.6 else 2) for s in scores]
    return disc[0] * 9 + disc[1] * 3 + disc[2]


def compute_reward(chosen, correct, result, difficulty='medium'):
    r = 0.0
    if chosen == correct:
        r += 1.0
        if difficulty == 'hard':
            r += 0.5
    else:
        r -= 0.5
        if difficulty == 'easy':
            r -= 0.3
    r += result.confidence * 0.3
    if result.execution_time_ms > 500:
        r -= 0.1
    return r


class QLearningOrchestrator:
    def __init__(self, n_states=27, n_actions=3,
                 learning_rate=0.15, discount_factor=0.9,
                 epsilon_start=1.0, epsilon_end=0.01, epsilon_decay=0.99):
        self.n_states = n_states
        self.n_actions = n_actions
        self.lr = learning_rate
        self.gamma = discount_factor
        self.epsilon = epsilon_start
        self.epsilon_end = epsilon_end
        self.epsilon_decay = epsilon_decay
        self.q_table = np.zeros((n_states, n_actions))
        self.history = {'rewards': [], 'accuracy': [], 'epsilon': []}

    def select_action(self, state, training=True):
        if training and random.random() < self.epsilon:
            return random.randint(0, self.n_actions - 1)
        return int(np.argmax(self.q_table[state]))

    def update(self, state, action, reward, next_state):
        best_next = np.max(self.q_table[next_state])
        td_target = reward + self.gamma * best_next
        self.q_table[state, action] += self.lr * (td_target - self.q_table[state, action])

    def train(self, dataset, n_episodes=300, verbose=True):
        for episode in range(n_episodes):
            random.shuffle(dataset)
            ep_reward, correct, total = 0.0, 0, 0
            for i, row in enumerate(dataset):
                q, label, diff = row['query'], row['correct_agent'], row.get('difficulty', 'medium')
                s = extract_state(q)
                a = self.select_action(s, training=True)
                chosen = ACTION_SPACE[a]
                res = AGENT_REGISTRY[chosen].execute(q)
                reward = compute_reward(chosen, label, res, diff)
                ns = extract_state(dataset[i+1]['query']) if i+1 < len(dataset) else s
                self.update(s, a, reward, ns)
                ep_reward += reward
                correct += int(chosen == label)
                total += 1
            self.epsilon = max(self.epsilon_end, self.epsilon * self.epsilon_decay)
            acc = correct / total
            self.history['rewards'].append(ep_reward)
            self.history['accuracy'].append(acc)
            self.history['epsilon'].append(self.epsilon)
            if verbose and (episode % 50 == 0 or episode == n_episodes - 1):
                print(f'  Ep {episode:4d} | Acc: {acc:.1%} | ε: {self.epsilon:.4f} | Reward: {ep_reward:.2f}')

    def predict_batch(self, rows):
        out = []
        for row in rows:
            q, label = row['query'], row['correct_agent']
            s = extract_state(q)
            a = self.select_action(s, training=False)
            chosen = ACTION_SPACE[a]
            res = AGENT_REGISTRY[chosen].execute(q)
            out.append({
                'query': q,
                'correct': label,
                'chosen': chosen,
                'ok': chosen == label,
                'difficulty': row.get('difficulty', '?'),
                'confidence': res.confidence,
                'exec_time_ms': res.execution_time_ms,
                'q_values': {ACTION_SPACE[i]: round(self.q_table[s, i], 3)
                             for i in range(self.n_actions)},
            })
        return out


print('Classes loaded.')

## 2. Train Full Dataset & Evaluate

In [ ]:
# Chuẩn bị full dataset từ CSV
full_dataset = df_orch[['query', 'correct_agent', 'difficulty']].to_dict('records')

random.seed(42)
random.shuffle(full_dataset)
split = int(0.8 * len(full_dataset))
train_set = full_dataset[:split]
test_set = full_dataset[split:]

print(f'Train: {len(train_set)} | Test: {len(test_set)}')

# Huấn luyện
orchestrator = QLearningOrchestrator(
    learning_rate=0.15, discount_factor=0.9,
    epsilon_start=1.0, epsilon_end=0.01, epsilon_decay=0.99,
)
orchestrator.train(train_set, n_episodes=500, verbose=True)

In [ ]:
# Predict trên test set
predictions = orchestrator.predict_batch(test_set)
df_pred = pd.DataFrame(predictions)
print(f'Test set predictions: {len(df_pred)} queries')
df_pred[['query', 'correct', 'chosen', 'ok', 'difficulty', 'confidence']].head(10)

## 3. Metric 1: Routing Accuracy

In [ ]:
overall_acc = df_pred['ok'].mean()
n_correct = df_pred['ok'].sum()
n_total = len(df_pred)

print('=== METRIC 1: ROUTING ACCURACY ===')
print(f'  Overall: {overall_acc:.1%}  ({n_correct}/{n_total})')

target_85 = overall_acc >= 0.85
print(f'  Target >85%: {"PASS" if target_85 else "FAIL"}  (actual: {overall_acc:.1%})')

In [ ]:
# By agent
print('--- Accuracy by Agent ---')
agent_targets = {'fundamental': 0.80, 'technical': 0.85, 'screening': 0.75}
acc_by_agent = df_pred.groupby('correct')['ok'].agg(['mean', 'sum', 'count'])
acc_by_agent.columns = ['accuracy', 'correct_n', 'total_n']

print(f'  {"Agent":<15} {"Accuracy":<12} {"Correct/Total":<15} {"Target":<10} {"Status"}')
print('  ' + '-' * 65)
for agent, row in acc_by_agent.iterrows():
    tgt = agent_targets.get(agent, 0.80)
    status = 'PASS' if row['accuracy'] >= tgt else 'FAIL'
    print(f'  {agent:<15} {row["accuracy"]:.1%}       '
          f'{int(row["correct_n"])}/{int(row["total_n"]):<13} '
          f'{tgt:.0%}       {status}')

In [ ]:
# By difficulty
print('--- Accuracy by Difficulty ---')
diff_targets = {'easy': 0.95, 'medium': 0.80, 'hard': 0.70}
acc_by_diff = df_pred.groupby('difficulty')['ok'].agg(['mean', 'sum', 'count'])
acc_by_diff.columns = ['accuracy', 'correct_n', 'total_n']

print(f'  {"Difficulty":<12} {"Accuracy":<12} {"Correct/Total":<15} {"Target":<10} {"Status"}')
print('  ' + '-' * 65)
for diff, row in acc_by_diff.iterrows():
    tgt = diff_targets.get(diff, 0.80)
    status = 'PASS' if row['accuracy'] >= tgt else 'FAIL'
    print(f'  {diff:<12} {row["accuracy"]:.1%}       '
          f'{int(row["correct_n"])}/{int(row["total_n"]):<13} '
          f'{tgt:.0%}       {status}')

## 4. Metric 2: Confusion Matrix

In [ ]:
print('=== METRIC 2: CONFUSION MATRIX ===')
confusion = pd.crosstab(
    df_pred['correct'], df_pred['chosen'],
    rownames=['Actual'], colnames=['Predicted']
)
# Ensure all columns present
for agent in ACTION_SPACE:
    if agent not in confusion.columns:
        confusion[agent] = 0
    if agent not in confusion.index:
        confusion.loc[agent] = 0
confusion = confusion[ACTION_SPACE].reindex(ACTION_SPACE, fill_value=0)
print(confusion.to_string())
print()
print('  Diagonal = correct predictions')
print('  Off-diagonal = routing errors')

In [ ]:
# Precision, Recall, F1 per agent
print('--- Precision / Recall / F1 per Agent ---')
print(f'  {"Agent":<15} {"Precision":<12} {"Recall":<12} {"F1":<10}')
print('  ' + '-' * 50)

for agent in ACTION_SPACE:
    tp = confusion.loc[agent, agent] if agent in confusion.index and agent in confusion.columns else 0
    fp = confusion[agent].sum() - tp if agent in confusion.columns else 0
    fn = confusion.loc[agent].sum() - tp if agent in confusion.index else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    print(f'  {agent:<15} {precision:.3f}        {recall:.3f}        {f1:.3f}')

## 5. Metric 3: Cumulative Reward & Training Curve

In [ ]:
print('=== METRIC 3: CUMULATIVE REWARD & TRAINING CURVE ===')
rewards = np.array(orchestrator.history['rewards'])
accuracies = np.array(orchestrator.history['accuracy'])
epsilons = np.array(orchestrator.history['epsilon'])

# Stats by window
windows = [(0, 50), (50, 150), (150, 300), (300, None)]
print(f'  {"Window":<20} {"Avg Reward":<15} {"Avg Acc":<12} {"Trend"}')
print('  ' + '-' * 60)

prev_acc = None
for start, end in windows:
    r_w = rewards[start:end]
    a_w = accuracies[start:end]
    if len(r_w) == 0:
        continue
    name = f'Ep {start}-{end if end else len(rewards)}'
    avg_acc = np.mean(a_w)
    trend = ''
    if prev_acc is not None:
        diff = avg_acc - prev_acc
        trend = f'+{diff:.1%}' if diff >= 0 else f'{diff:.1%}'
    print(f'  {name:<20} {np.mean(r_w):.2f} ± {np.std(r_w):.2f}    '
          f'{avg_acc:.1%} ± {np.std(a_w):.1%}  {trend}')
    prev_acc = avg_acc

# Cumulative reward
cum_reward = np.cumsum(rewards)
print(f'\n  Total cumulative reward: {cum_reward[-1]:.2f}')
print(f'  Avg reward per episode:  {rewards.mean():.2f} ± {rewards.std():.2f}')

## 6. Metric 4: Convergence Analysis

In [ ]:
print('=== METRIC 4: CONVERGENCE ANALYSIS ===')
thresholds = [0.60, 0.70, 0.75, 0.80, 0.85, 0.90]

print(f'  {"Threshold":<12} {"First Episode":<15} {"Target (<200)":<15} {"Status"}')
print('  ' + '-' * 55)
for thr in thresholds:
    conv = next((i for i, a in enumerate(accuracies) if a >= thr), None)
    target = 200
    if conv is not None:
        status = 'PASS' if conv < target else 'LATE'
        print(f'  {thr:.0%}         Ep {conv:<13} {target:<15} {status}')
    else:
        print(f'  {thr:.0%}         NOT REACHED    {target:<15} FAIL')

In [ ]:
# Rolling average accuracy
print('--- Rolling 50-ep Average Accuracy ---')
window = 50
rolling = pd.Series(accuracies).rolling(window).mean()

# Print at key milestones
milestones = [50, 100, 150, 200, 300, 400, 499]
print(f'  {"Episode":<10} {"Rolling Avg"}')
for ep in milestones:
    if ep < len(rolling) and not np.isnan(rolling.iloc[ep]):
        print(f'  {ep:<10} {rolling.iloc[ep]:.1%}')

final_window = accuracies[-50:]
print(f'\n  Last 50 episodes: {np.mean(final_window):.1%} ± {np.std(final_window):.1%}')

## 7. Metric 5: Q-Value Stability

In [ ]:
print('=== METRIC 5: Q-VALUE STABILITY ===')
q = orchestrator.q_table

# Active states
active_mask = np.abs(q).max(axis=1) > 0.001
n_active = active_mask.sum()
print(f'  Active states: {n_active} / {orchestrator.n_states}')
print()

# Q-value statistics
q_active = q[active_mask]
print('--- Q-Table Statistics (active states) ---')
print(f'  Q mean: {q_active.mean():.4f}')
print(f'  Q std:  {q_active.std():.4f}')
print(f'  Q min:  {q_active.min():.4f}')
print(f'  Q max:  {q_active.max():.4f}')
print()

# Per-action Q-value means
print('--- Q-value mean per action (active states) ---')
for i, agent in enumerate(ACTION_SPACE):
    q_vals = q_active[:, i]
    print(f'  {agent:<15}: mean={q_vals.mean():.3f}  std={q_vals.std():.3f}  '
          f'max={q_vals.max():.3f}')

In [ ]:
# Q-table as DataFrame
print('--- Q-Table (non-zero states) ---')
df_q = pd.DataFrame(q, columns=ACTION_SPACE)
df_q.index.name = 'state'
df_q['best_action'] = df_q.idxmax(axis=1)
df_q['q_range'] = df_q[ACTION_SPACE].max(axis=1) - df_q[ACTION_SPACE].min(axis=1)

df_q_active = df_q[df_q[ACTION_SPACE].abs().max(axis=1) > 0.001].copy()
df_q_active = df_q_active.round(3)
print(f'  Showing {len(df_q_active)} active states out of {orchestrator.n_states}')
df_q_active.sort_values('q_range', ascending=False)

## 8. Metric 6: Confidence & Latency Analysis

In [ ]:
print('=== METRIC 6: CONFIDENCE & LATENCY ===')

# Confidence by correctness
print('--- Agent Confidence ---')
print(f'  Correct predictions:   avg conf = {df_pred[df_pred["ok"]]["confidence"].mean():.3f}')
print(f'  Incorrect predictions: avg conf = {df_pred[~df_pred["ok"]]["confidence"].mean():.3f}')
print()

# Confidence by agent
print('--- Confidence by Agent ---')
conf_agent = df_pred.groupby('correct')['confidence'].agg(['mean', 'std', 'min', 'max']).round(3)
print(conf_agent.to_string())
print()

# Latency
print('--- Execution Latency (ms) ---')
lat = df_pred['exec_time_ms']
print(f'  Mean: {lat.mean():.1f} ms')
print(f'  P50:  {lat.quantile(0.50):.1f} ms')
print(f'  P95:  {lat.quantile(0.95):.1f} ms')
print(f'  Max:  {lat.max():.1f} ms')
print(f'  Slow (>500ms): {(lat > 500).sum()} ({(lat > 500).mean()*100:.1f}%)')

target_latency_p95 = 2000  # 2 seconds
p95 = lat.quantile(0.95)
print(f'\n  Target P95 < {target_latency_p95}ms: {"PASS" if p95 < target_latency_p95 else "FAIL"}  (actual: {p95:.0f}ms)')

## 9. Metric 7: Epsilon Decay & Exploration Efficiency

In [ ]:
print('=== METRIC 7: EXPLORATION ANALYSIS ===')

# Epsilon decay milestones
eps_arr = np.array(orchestrator.history['epsilon'])
print('--- Epsilon Decay ---')
milestones = [0, 50, 100, 150, 200, 300, 400, len(eps_arr)-1]
print(f'  {"Episode":<10} {"Epsilon"}')
for ep in milestones:
    if ep < len(eps_arr):
        print(f'  {ep:<10} {eps_arr[ep]:.4f}')

print()
# Exploration efficiency: reward per episode relative to random
# Random agent expected accuracy = 1/3 ≈ 33.3%
random_baseline_acc = 1 / N_ACTIONS
final_acc = np.mean(accuracies[-50:])
improvement = (final_acc - random_baseline_acc) / random_baseline_acc
print(f'--- Exploration Efficiency ---')
print(f'  Random baseline accuracy: {random_baseline_acc:.1%}')
print(f'  Final accuracy (last 50): {final_acc:.1%}')
print(f'  Relative improvement over random: +{improvement:.1%}')

# First time beating 2x random
target_2x = random_baseline_acc * 2
ep_2x = next((i for i, a in enumerate(accuracies) if a >= target_2x), None)
print(f'\n  First episode beating 2x random ({target_2x:.1%}): Ep {ep_2x}')

## 10. Scorecard Tổng Kết

In [ ]:
print('=' * 70)
print('SCORECARD TỔNG KẾT – RL ORCHESTRATOR PoC')
print('=' * 70)

# Collect all metrics
final_test_acc = overall_acc
conv_85 = next((i for i, a in enumerate(accuracies) if a >= 0.85), None)
last50_acc = np.mean(accuracies[-50:])
last50_std = np.std(accuracies[-50:])
p95_lat = df_pred['exec_time_ms'].quantile(0.95)

metrics_table = [
    ('Routing Accuracy (test)',    f'{final_test_acc:.1%}',
     '> 85%',  'PASS' if final_test_acc >= 0.85 else 'FAIL'),
    ('Last 50 ep Accuracy',        f'{last50_acc:.1%} ± {last50_std:.1%}',
     '> 80%',  'PASS' if last50_acc >= 0.80 else 'FAIL'),
    ('Convergence to 85%',
     f'Ep {conv_85}' if conv_85 else 'NOT REACHED',
     '< 200',  'PASS' if conv_85 and conv_85 < 200 else 'FAIL'),
    ('Fundamental accuracy',
     f'{acc_by_agent.loc["fundamental", "accuracy"]:.1%}' if "fundamental" in acc_by_agent.index else 'N/A',
     '> 80%',  'PASS' if "fundamental" in acc_by_agent.index and acc_by_agent.loc["fundamental", "accuracy"] >= 0.80 else 'FAIL'),
    ('Technical accuracy',
     f'{acc_by_agent.loc["technical", "accuracy"]:.1%}' if "technical" in acc_by_agent.index else 'N/A',
     '> 85%',  'PASS' if "technical" in acc_by_agent.index and acc_by_agent.loc["technical", "accuracy"] >= 0.85 else 'FAIL'),
    ('Screening accuracy',
     f'{acc_by_agent.loc["screening", "accuracy"]:.1%}' if "screening" in acc_by_agent.index else 'N/A',
     '> 75%',  'PASS' if "screening" in acc_by_agent.index and acc_by_agent.loc["screening", "accuracy"] >= 0.75 else 'FAIL'),
    ('P95 Latency',                f'{p95_lat:.0f} ms',
     '< 2000ms', 'PASS' if p95_lat < 2000 else 'FAIL'),
    ('Active Q-states',            f'{n_active} / {orchestrator.n_states}',
     '> 5 states', 'PASS' if n_active > 5 else 'FAIL'),
]

print(f'  {"Metric":<30} {"Value":<22} {"Target":<15} {"Status"}')
print('  ' + '-' * 80)
for metric, value, target, status in metrics_table:
    icon = '' if status == 'PASS' else ''
    print(f'  {metric:<30} {value:<22} {target:<15} {status}')

passed = sum(1 for _, _, _, s in metrics_table if s == 'PASS')
total_m = len(metrics_table)
print()
print(f'  TOTAL: {passed}/{total_m} metrics passed ({passed/total_m:.0%})')
print()

# Known issues summary
print('--- Known Issues ---')
issues = [
    'Screening agent thấp nhất do keyword overlap với fundamental queries',
    'Hard queries (mixed-intent) cần multi-agent chaining (Phase 3)',
    'State space 27 còn thô – Phase 2 dùng DQN + sentence embeddings',
    'Synthetic data – Phase 2 cần real API (VNDirect, SSI)',
]
for issue in issues:
    print(f'  ! {issue}')

## 11. So Sánh Với Baseline Random

In [ ]:
print('=== SO SÁNH VỚI BASELINE ===')

# Random baseline
random.seed(99)
random_preds = [random.choice(ACTION_SPACE) for _ in test_set]
random_correct = [pred == row['correct_agent'] for pred, row in zip(random_preds, test_set)]
random_acc = np.mean(random_correct)

# Majority baseline (always predict most frequent class)
majority_class = df_orch['correct_agent'].value_counts().idxmax()
majority_correct = [majority_class == row['correct_agent'] for row in test_set]
majority_acc = np.mean(majority_correct)

print(f'  {"Model":<30} {"Accuracy"}')
print('  ' + '-' * 42)
print(f'  {"Random Baseline":<30} {random_acc:.1%}')
print(f'  {"Majority Class Baseline":<30} {majority_acc:.1%} (always "{majority_class}")')
print(f'  {"Q-Learning Orchestrator":<30} {final_test_acc:.1%}')
print()
print(f'  Improvement vs random:   +{(final_test_acc - random_acc)*100:.1f}pp')
print(f'  Improvement vs majority: +{(final_test_acc - majority_acc)*100:.1f}pp')